In [1]:
!pip install -q numpy scikit-learn rouge-score nltk

  Preparing metadata (setup.py) ... done


In [2]:
import numpy as np
import nltk
nltk.download("punkt", quiet=True)

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    roc_auc_score,
    average_precision_score,
)
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

In [3]:
# Text Generation Outputs
foundation_model_outputs = [
    "The cat sat on the mat.",
    "Code generation is fun.",
    "The image shows a dog.",
]

fine_tuned_model_outputs = [
    "The feline rested on the rug.",
    "Generating code is enjoyable.",
    "A dog is visible in the picture.",
]

reference_outputs = [
    "The cat sat on the mat.",
    "Code generation is fun.",
    "A dog is in the image.",
]

# Binary Classification Predictions
foundation_model_predictions_binary = np.array([0, 1, 0, 1])
fine_tuned_model_predictions_binary = np.array([1, 1, 0, 1])
reference_labels_binary             = np.array([1, 1, 0, 0])

# Probability Predictions
foundation_model_predictions_prob = np.array([0.2, 0.8, 0.1, 0.9])
fine_tuned_model_predictions_prob = np.array([0.7, 0.9, 0.3, 0.8])

In [4]:
def evaluate_text_generation(foundation_outputs, fine_tuned_outputs, reference_outputs):
    """
    Evaluates text generation quality using BLEU and ROUGE scores.
    """
    bleu_scores_foundation = []
    bleu_scores_finetuned  = []
    rouge_scores_foundation = []
    rouge_scores_finetuned  = []

    scorer             = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
    smoothing_function = SmoothingFunction().method4

    for f_out, ft_out, ref_out in zip(foundation_outputs, fine_tuned_outputs, reference_outputs):
        bleu_scores_foundation.append(
            sentence_bleu([ref_out.split()], f_out.split(),
                          smoothing_function=smoothing_function)
        )
        bleu_scores_finetuned.append(
            sentence_bleu([ref_out.split()], ft_out.split(),
                          smoothing_function=smoothing_function)
        )
        rouge_scores_foundation.append(scorer.score(ref_out, f_out))
        rouge_scores_finetuned.append(scorer.score(ref_out, ft_out))

    avg_bleu_foundation = np.mean(bleu_scores_foundation)
    avg_bleu_finetuned  = np.mean(bleu_scores_finetuned)

    rouge1_f_avg  = np.mean([s["rouge1"].fmeasure for s in rouge_scores_foundation])
    rougeL_f_avg  = np.mean([s["rougeL"].fmeasure for s in rouge_scores_foundation])
    rouge1_ft_avg = np.mean([s["rouge1"].fmeasure for s in rouge_scores_finetuned])
    rougeL_ft_avg = np.mean([s["rougeL"].fmeasure for s in rouge_scores_finetuned])

    print("="*55)
    print("   TEXT GENERATION EVALUATION")
    print("="*55)
    print(f"  Avg BLEU   (Foundation) : {avg_bleu_foundation:.4f}")
    print(f"  Avg BLEU   (Fine-tuned) : {avg_bleu_finetuned:.4f}")
    print(f"  Avg ROUGE-1 (Foundation): {rouge1_f_avg:.4f}")
    print(f"  Avg ROUGE-L (Foundation): {rougeL_f_avg:.4f}")
    print(f"  Avg ROUGE-1 (Fine-tuned): {rouge1_ft_avg:.4f}")
    print(f"  Avg ROUGE-L (Fine-tuned): {rougeL_ft_avg:.4f}")

In [5]:
def evaluate_classification_binary(foundation_preds, fine_tuned_preds, reference_labels):
    """
    Evaluates binary classification using accuracy, precision, recall, and F1.
    """
    print("\n" + "="*55)
    print("   BINARY CLASSIFICATION EVALUATION")
    print("="*55)
    print(f"  Foundation Accuracy  : {accuracy_score(reference_labels, foundation_preds):.4f}")
    print(f"  Fine-tuned Accuracy  : {accuracy_score(reference_labels, fine_tuned_preds):.4f}")
    print(f"  Foundation Precision : {precision_score(reference_labels, foundation_preds):.4f}")
    print(f"  Fine-tuned Precision : {precision_score(reference_labels, fine_tuned_preds):.4f}")
    print(f"  Foundation Recall    : {recall_score(reference_labels, foundation_preds):.4f}")
    print(f"  Fine-tuned Recall    : {recall_score(reference_labels, fine_tuned_preds):.4f}")
    print(f"  Foundation F1-Score  : {f1_score(reference_labels, foundation_preds):.4f}")
    print(f"  Fine-tuned F1-Score  : {f1_score(reference_labels, fine_tuned_preds):.4f}")

In [6]:
def evaluate_classification_prob(foundation_preds, fine_tuned_preds, reference_labels):
    """
    Evaluates classification using ROC-AUC and Average Precision scores.
    """
    print("\n" + "="*55)
    print("   PROBABILITY CLASSIFICATION EVALUATION")
    print("="*55)
    try:
        print(f"  Foundation ROC-AUC          : {roc_auc_score(reference_labels, foundation_preds):.4f}")
        print(f"  Fine-tuned ROC-AUC          : {roc_auc_score(reference_labels, fine_tuned_preds):.4f}")
        print(f"  Foundation Avg Precision    : {average_precision_score(reference_labels, foundation_preds):.4f}")
        print(f"  Fine-tuned Avg Precision    : {average_precision_score(reference_labels, fine_tuned_preds):.4f}")
    except ValueError:
        print("  ROC-AUC cannot be calculated when only one class is present.")


In [7]:
evaluate_text_generation(
    foundation_model_outputs,
    fine_tuned_model_outputs,
    reference_outputs
)

evaluate_classification_binary(
    foundation_model_predictions_binary,
    fine_tuned_model_predictions_binary,
    reference_labels_binary
)

evaluate_classification_prob(
    foundation_model_predictions_prob,
    fine_tuned_model_predictions_prob,
    reference_labels_binary
)

print("\n" + "="*55)
print("   Evaluation Completed Successfully!")
print("="*55)

   TEXT GENERATION EVALUATION
  Avg BLEU   (Foundation) : 0.6667
  Avg BLEU   (Fine-tuned) : 0.1371
  Avg ROUGE-1 (Foundation): 0.9091
  Avg ROUGE-L (Foundation): 0.7879
  Avg ROUGE-1 (Fine-tuned): 0.6731
  Avg ROUGE-L (Fine-tuned): 0.5897

   BINARY CLASSIFICATION EVALUATION
  Foundation Accuracy  : 0.5000
  Fine-tuned Accuracy  : 0.7500
  Foundation Precision : 0.5000
  Fine-tuned Precision : 0.6667
  Foundation Recall    : 0.5000
  Fine-tuned Recall    : 1.0000
  Foundation F1-Score  : 0.5000
  Fine-tuned F1-Score  : 0.8000

   PROBABILITY CLASSIFICATION EVALUATION
  Foundation ROC-AUC          : 0.5000
  Fine-tuned ROC-AUC          : 0.7500
  Foundation Avg Precision    : 0.5833
  Fine-tuned Avg Precision    : 0.8333

   Evaluation Completed Successfully!
